# 01. Knowledge-based Agents dan Wumpus World


Notebook ini memperkenalkan apa itu agent berbasis pengetahuan dan dunia yang
dipakai sebagai contoh sepanjang materi, yaitu Wumpus World. Belum ada logika
formal di sini. Simbol seperti $P_{1,2}$ mulai digunakan pada Notebook 02,
sedangkan aturan $R_1$ sampai $R_5$ baru dibangun pada Notebook 04.

**Batasan:** jangan pakai `Expr`, `expr()`, atau operator `|'==>'|`. Itu baru
diperkenalkan di Notebook 03. Contoh kode di sini pakai Python biasa saja.
`psource()` boleh dipakai untuk menampilkan source code.

**Output yang diharapkan:** pembaca paham kenapa agent butuh knowledge base, dan
hafal aturan main Wumpus World tanpa perlu buka slide lagi.

## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel mana pun yang lain.

Sel ini memasang dependensi yang diperlukan, mencari folder yang berisi
`logic.py` dan `utils.py`, lalu mengimpornya. Kalau notebook dibuka lewat Google
Colab, repo akan di-clone otomatis. Tidak ada yang perlu diubah di sini.

Environment sudah siap kalau baris terakhir output mencetak
`Check       : tt_entails(P & Q, Q) = True`.

In [ ]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

---
# 1.1 Knowledge-based Agents

*Logical agent*, adalah agent yang dapat mengambil keputusan berdasarkan logika dan penalaran.

*knowledge-based agent* (KB agent), adalah agent yang
menentukan aksi dengan menyimpan pengetahuan tentang dunia dan menalar dari
pengetahuan tersebut. 

Pendekatan ini memiliki tiga kemampuan utama:
1. membuat representasi dari dunia yang kompleks;
2. melakukan *inference* untuk menghasilkan representasi baru; dan
3. menyimpulkan aksi berdasarkan hasil representasi tersebut.

Karena pengetahuannya dinyatakan secara eksplisit, KB agent dapat menerima tugas
baru sebagai *goal*, memperoleh kompetensi dengan cepat melalui informasi yang
diberikan atau dipelajari, dan beradaptasi saat environment berubah dengan
memperbarui pengetahuan yang relevan.

Komponen utama dari **knowledge base (KB)** adalah himpunan *sentence*.
Sebuah *sentence* menyatakan suatu fakta atau aturan dalam **knowledge
representation language**, yakni bahasa yang menentukan bentuk pengetahuan yang
boleh disimpan. Dua operasi dasarnya adalah:

- **TELL**: menambahkan *sentence* baru ke KB;
- **ASK**: menanyakan apa yang dapat disimpulkan dari KB.

TELL dan ASK dapat melibatkan *inference*, yaitu proses menurunkan *sentence*
baru dari *sentence* yang sudah ada. Berbeda dari agent yang hanya mencocokkan
setiap percept ke aksi dalam tabel tetap, KB agent memisahkan isi pengetahuan
dari mekanisme program. Akibatnya, fakta atau aturan dapat ditambah tanpa harus
menulis ulang seluruh pengendali agent.

## Contoh penerapan

Bayangkan KB mula-mula kosong. Agent kemudian melakukan TELL terhadap tiga
*sentence* berikut dalam bahasa biasa:

1. Kotak yang memiliki *breeze* bersebelahan dengan sedikitnya satu pit.
2. Agent merasakan *breeze* di `[2,1]`.
3. Kotak `[1,1]` tidak berisi pit.

Saat agent melakukan ASK, "Apakah sedikitnya satu dari `[2,2]` dan `[3,1]`
berisi pit?", KB dapat menjawab ya melalui *inference*. Namun KB belum dapat
memastikan kotak yang mana. Contoh ini menunjukkan perbedaan antara fakta yang
dimasukkan dengan TELL dan pengetahuan baru yang diperoleh melalui ASK. Bentuk
logika formalnya akan dibahas pada notebook berikutnya.

---
# 1.2 Arsitektur KB Agent

Figure 7.1 memperlihatkan siklus kerja generik sebuah KB agent. Variabel
persisten `KB` mempertahankan pengetahuan antarpemanggilan, sedangkan pencacah
waktu `t` mula-mula bernilai 0. Pada setiap percept baru, agent melakukan tiga
langkah utama:

1. `TELL(KB, MAKE-PERCEPT-SENTENCE(percept, t))` mengubah percept menjadi
   *sentence* bertanda waktu, kemudian menyimpannya.
2. `ASK(KB, MAKE-ACTION-QUERY(t))` meminta aksi yang seharusnya dilakukan pada
   waktu tersebut. Jawaban diperoleh dari pengetahuan dan proses inferensi.
3. `TELL(KB, MAKE-ACTION-SENTENCE(action, t))` merekam aksi yang benar-benar
   dipilih sebelum aksi dikembalikan ke environment.

Ketiga fungsi `MAKE-...` menjembatani data mentah program dengan *sentence*
yang dipahami KB. Pencacah `t` mencegah dua percept atau aksi pada waktu berbeda
dianggap sebagai kejadian yang sama. Riwayat aksi juga perlu di-TELL karena
state berikutnya bergantung pada aksi sebelumnya: percept *bump*, misalnya, baru
bermakna jika KB mengetahui bahwa agent baru saja mencoba maju.

![Figure 7.1: generic knowledge-based agent](img/fig-7-1-kb-agent.png)

## Contoh penerapan

Cell pertama menampilkan implementasi `KBAgentProgram` dari `logic.py`. Fokuskan
perhatian pada urutan TELL–ASK–TELL; pembentuk *sentence* formal yang muncul di
source code baru akan dipelajari pada Notebook 03. Cell berikutnya membuat
versi Python biasa agar siklus yang sama dapat langsung dijalankan tanpa bahasa
logika formal.

In [ ]:
psource(KBAgentProgram)

In [ ]:
class ListKnowledgeBase:
    """A minimal KB that stores time-stamped percepts and actions."""

    def __init__(self, action_rule):
        self.sentences = []
        self.action_rule = action_rule

    def tell(self, sentence):
        self.sentences.append(sentence)

    def ask(self, query):
        time = query["time"]
        percept = next(
            sentence["value"]
            for sentence in reversed(self.sentences)
            if sentence["kind"] == "percept" and sentence["time"] == time
        )
        return self.action_rule(percept)


def make_plain_kb_agent_program(kb):
    """Implement the TELL-ASK-TELL cycle without formal logic syntax."""
    steps = itertools.count()

    def program(percept):
        time = next(steps)
        kb.tell({"kind": "percept", "time": time, "value": percept})
        action = kb.ask({"kind": "action", "time": time})
        kb.tell({"kind": "action", "time": time, "value": action})
        return action

    return program


def demo_action_rule(percept):
    return "Grab" if percept.get("glitter") else "Forward"


demo_kb = ListKnowledgeBase(demo_action_rule)
demo_agent = make_plain_kb_agent_program(demo_kb)
print("Action at t=0:", demo_agent({"glitter": False}))
print("Action at t=1:", demo_agent({"glitter": True}))
print("KB sentences   :", demo_kb.sentences)

Pada keluaran tersebut, setiap pemanggilan menghasilkan dua catatan KB: satu
percept dan satu aksi dengan nilai waktu yang sama. Perubahan aksi pada `t=1`
terjadi karena ASK menggunakan percept terbaru yang sudah di-TELL.

---
# 1.3 Wumpus World

Wumpus World adalah gua berbentuk jaringan ruangan yang dihubungkan oleh lorong.
Dunia ini dipakai untuk memperlihatkan bagaimana agent bertindak ketika isi
ruangan tidak terlihat secara langsung. Tiga objek utamanya adalah:

- **Wumpus**, monster yang memakan agent yang memasuki ruangannya. Wumpus dapat
  ditembak, tetapi agent hanya mempunyai satu anak panah.
- **Pit**, lubang tanpa dasar yang menjebak agent yang memasukinya.
- **Gold**, tujuan bernilai positif yang membuat risiko memasuki gua layak
  dipertimbangkan.

Figure 7.2 menggunakan grid $4 \times 4$. Koordinat ditulis `[x,y]`: nilai `x`
bertambah ke kanan dan `y` bertambah ke atas. Agent selalu mulai di `[1,1]`,
yaitu sudut kiri bawah, sambil menghadap ke kanan. Pada contoh ini Wumpus berada
di `[1,3]`, emas di `[2,3]`, dan pit di `[3,1]`, `[3,3]`, serta `[4,4]`.

![Figure 7.2: sebuah Wumpus World](img/fig-7-2-wumpus-world.png)

## Contoh penerapan

Peta di bawah disimpan sebagai `dict` dan `set`, lalu ditampilkan sebagai grid.
Representasi ini belum memuat aturan logika; fungsinya adalah menjadi state
environment yang akan dibaca oleh fungsi sensor.

In [ ]:
wumpus_world = {
    "size": 4,
    "start": (1, 1),
    "start_direction": "right",
    "wumpus": (1, 3),
    "pits": {(3, 1), (3, 3), (4, 4)},
    "gold": (2, 3),
}


def square_label(world, position):
    """Return a compact label for one square in the world."""
    labels = []
    if position == world["start"]:
        labels.append("A>")
    if position == world["wumpus"]:
        labels.append("W")
    if position in world["pits"]:
        labels.append("P")
    if position == world["gold"]:
        labels.append("G")
    return "+".join(labels) if labels else "."

In [ ]:
def render_world(world):
    """Build a table whose layout follows the coordinates in Figure 7.2."""
    size = world["size"]
    rows = [
        [square_label(world, (x, y)) for x in range(1, size + 1)]
        for y in range(size, 0, -1)
    ]
    return pd.DataFrame(
        rows,
        index=[f"y={y}" for y in range(size, 0, -1)],
        columns=[f"x={x}" for x in range(1, size + 1)],
    )


render_world(wumpus_world)

Label `A>` menandai agent di `[1,1]` yang menghadap kanan; `W`, `P`, dan `G`
masing-masing menandai Wumpus, pit, dan gold. Titik berarti kotak tersebut tidak
berisi objek utama, bukan berarti agent otomatis telah mengetahui bahwa kotak itu
aman.

---
# 1.4 PEAS Definition

PEAS merangkum rancangan tugas agent melalui **Performance measure**,
**Environment**, **Actuators**, dan **Sensors**. Definisi Wumpus World adalah:

| Komponen | Definisi |
|---|---|
| Performance measure | `+1000` jika memperoleh gold; `-1000` jika jatuh ke pit atau dimakan Wumpus; `-1` untuk setiap aksi; dan `-10` ketika memakai anak panah. Permainan selesai saat agent mati atau berhasil melakukan Climb keluar dari gua. |
| Environment | Grid $4 \times 4$. Agent selalu mulai di `[1,1]` menghadap kanan. Gold dan Wumpus ditempatkan secara acak dengan distribusi uniform pada kotak selain kotak awal. Setiap kotak selain kotak awal dapat berisi pit dengan probabilitas 0,2. |
| Actuators | `Forward`, `TurnLeft` 90 derajat, `TurnRight` 90 derajat, `Grab`, `Shoot`, dan `Climb`. Agent mati jika memasuki pit atau kotak Wumpus hidup. `Forward` ke dinding tidak memindahkan agent, `Shoot` melontarkan satu-satunya anak panah lurus sesuai arah hadap, dan `Climb` hanya berhasil dari `[1,1]`. |
| Sensors | `Stench` pada kotak Wumpus dan kotak yang bersebelahan langsung; `Breeze` pada kotak yang bersebelahan langsung dengan pit; `Glitter` pada kotak gold; `Bump` setelah menabrak dinding; serta `Scream` di seluruh gua saat Wumpus mati. |

Setiap percept disusun dengan urutan tetap:

```text
[Stench, Breeze, Glitter, Bump, Scream]
```

`None` digunakan jika sensor tidak aktif. Istilah *bersebelahan langsung* hanya
mencakup atas, bawah, kiri, dan kanan—bukan diagonal. Batas ini penting karena
menentukan kotak mana yang boleh menjadi kandidat pit atau Wumpus ketika agent
menalar.

## Contoh penerapan

Fungsi berikut menghitung tetangga ortogonal dan percept dari peta Figure 7.2.
`Bump` dan `Scream` diberikan sebagai parameter karena keduanya adalah kejadian,
bukan sifat tetap suatu kotak.

In [ ]:
PERCEPT_ORDER = ["Stench", "Breeze", "Glitter", "Bump", "Scream"]


def orthogonal_neighbors(position, size):
    """Return in-bounds horizontal and vertical neighbors."""
    x, y = position
    candidates = [(x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)]
    return [
        candidate
        for candidate in candidates
        if 1 <= candidate[0] <= size and 1 <= candidate[1] <= size
    ]

In [ ]:
def get_percept(world, position, bump=False, scream=False):
    """Return [Stench, Breeze, Glitter, Bump, Scream] at a position."""
    size = world["size"]
    x, y = position
    if not (1 <= x <= size and 1 <= y <= size):
        raise ValueError(f"Position {position} is outside the {size}x{size} world")

    neighbors = set(orthogonal_neighbors(position, size))
    has_stench = position == world["wumpus"] or world["wumpus"] in neighbors
    has_breeze = bool(neighbors & world["pits"])
    has_glitter = position == world["gold"]

    active = [has_stench, has_breeze, has_glitter, bump, scream]
    return [name if is_active else None for name, is_active in zip(PERCEPT_ORDER, active)]

In [ ]:
sample_positions = [(1, 1), (2, 1), (1, 2)]
sample_percepts = {
    f"[{x},{y}]": get_percept(wumpus_world, (x, y))
    for x, y in sample_positions
}

assert sample_percepts["[1,1]"] == [None, None, None, None, None]
assert sample_percepts["[2,1]"] == [None, "Breeze", None, None, None]
assert sample_percepts["[1,2]"] == ["Stench", None, None, None, None]

pd.DataFrame.from_dict(sample_percepts, orient="index", columns=PERCEPT_ORDER)

Hasilnya sama dengan Figure 7.2: `[1,1]` tidak menerima sinyal, `[2,1]`
menerima Breeze dari pit `[3,1]`, dan `[1,2]` menerima Stench dari Wumpus
`[1,3]`. Pit diagonal tidak menyalakan Breeze.

---
# 1.5 Karakteristik Environment dan Penjelajahan Agent

Wumpus World memiliki lima karakteristik berikut:

- **Discrete**: kotak, waktu, percept, dan pilihan aksi dapat dibedakan sebagai
  unit-unit terpisah.
- **Static**: keadaan gua tidak berubah sendiri ketika agent sedang menalar.
- **Single-agent**: hanya penjelajah yang mengambil keputusan; Wumpus merupakan
  bagian dari environment, bukan agent lawan yang menyusun strategi.
- **Sequential**: sebuah aksi memengaruhi situasi dan pilihan aksi berikutnya.
  Reward dapat baru diperoleh setelah rangkaian panjang aksi, sehingga keputusan
  tidak boleh dinilai sebagai kejadian terpisah.
- **Partially observable**: agent tidak melihat seluruh state secara langsung.
  Posisi agent, status hidup Wumpus, dan ketersediaan anak panah adalah contoh
  aspek state yang tidak diberikan langsung oleh sensor. Agent harus melacak atau
  menyimpulkannya dari percept dan riwayat aksi.

Karakteristik terakhir menjelaskan mengapa agent memerlukan KB: satu percept
lokal harus digabungkan dengan percept sebelumnya untuk membentuk gambaran dunia.

### Trace Figure 7.3 dan Figure 7.4

1. **Kondisi awal di `[1,1]`.** Percept
   `[None, None, None, None, None]` berarti tidak ada Breeze maupun Stench.
   Karena itu, kotak yang bersebelahan langsung, `[1,2]` dan `[2,1]`, tidak
   mengandung pit atau Wumpus dan ditandai OK. Agent aman bergerak ke `[2,1]`.
2. **Setelah langkah pertama di `[2,1]`.** Percept
   `[None, Breeze, None, None, None]` menunjukkan sedikitnya satu pit berada di
   sekitar `[2,1]`. Setelah `[1,1]` disisihkan, kandidatnya adalah `[2,2]` atau
   `[3,1]`. Agent belum boleh memilih salah satunya secara sembarang, sehingga
   kembali melalui kotak aman dan menjelajah `[1,2]`.
3. **Setelah langkah ketiga di `[1,2]`.** Percept
   `[Stench, None, None, None, None]` meniadakan pit di `[1,3]` dan `[2,2]`.
   Akibatnya, Breeze terdahulu hanya dapat dijelaskan oleh pit di `[3,1]`.
   Stench di sini membuat `[1,3]` atau `[2,2]` menjadi kandidat Wumpus, tetapi
   tidak adanya Stench di `[2,1]` menyingkirkan `[2,2]`. Jadi Wumpus pasti berada
   di `[1,3]`, walaupun kotak itu belum pernah dikunjungi. `[2,2]` kini OK.
4. **Setelah langkah kelima di `[2,3]`.** Setelah melewati `[2,2]`, agent menerima
   `[Stench, Breeze, Glitter, None, None]`. Glitter menyatakan gold berada di
   kotak yang sama, sehingga aksi yang masuk akal adalah Grab lalu kembali melalui
   jalur yang sudah diketahui aman. Breeze juga menunjukkan kandidat pit baru di
   `[2,4]` atau `[3,3]`, tetapi gold dapat diambil tanpa memasuki keduanya.

![Figure 7.3: dua tahap awal penjelajahan](img/fig-7-3-langkah-pertama.png)

![Figure 7.4: dua tahap lanjutan penjelajahan](img/fig-7-4-langkah-lanjutan.png)

Jika informasi yang tersedia benar dan aturan inferensinya valid, setiap
kesimpulan logis di atas dijamin benar. Jaminan ini merupakan sifat mendasar
*logical reasoning*; agent tidak sekadar memilih kotak yang tampak paling mungkin.

## Contoh penerapan

Cell berikut memakai `get_percept()` untuk mereproduksi empat snapshot pada
Figure 7.3 dan 7.4. Kesimpulan ditulis sebagai string agar fokus tetap pada alur
berpikir; inferensi otomatis baru dibangun pada notebook berikutnya.

In [ ]:
exploration_trace = [
    {
        "move": 0,
        "position": (1, 1),
        "conclusion": "[1,2] dan [2,1] aman; bergerak ke [2,1].",
    },
    {
        "move": 1,
        "position": (2, 1),
        "conclusion": "Pit ada di [2,2] atau [3,1]; jangan memasuki keduanya.",
    },
    {
        "move": 3,
        "position": (1, 2),
        "conclusion": "Wumpus ada di [1,3], pit ada di [3,1], dan [2,2] aman.",
    },
    {
        "move": 5,
        "position": (2, 3),
        "conclusion": "Gold ada di sini; Grab lalu pulang melalui jalur aman.",
    },
]

In [ ]:
trace_rows = []
for state in exploration_trace:
    x, y = state["position"]
    percept = get_percept(wumpus_world, state["position"])
    trace_rows.append(
        {
            "Move": state["move"],
            "Position": f"[{x},{y}]",
            "Percept": percept,
            "Kesimpulan": state["conclusion"],
        }
    )

pd.DataFrame(trace_rows)

In [ ]:
expected_percepts = [
    [None, None, None, None, None],
    [None, "Breeze", None, None, None],
    ["Stench", None, None, None, None],
    ["Stench", "Breeze", "Glitter", None, None],
]
actual_percepts = [row["Percept"] for row in trace_rows]
assert actual_percepts == expected_percepts
print("All trace percepts match Figures 7.3 and 7.4.")

Keempat percept cocok dengan caption Figure 7.3 dan Figure 7.4. Perhatikan bahwa
program sensor hanya menghasilkan observasi lokal; kolom Kesimpulan menunjukkan
pengetahuan yang diperoleh ketika observasi sekarang digabungkan dengan riwayat.

---
# Latihan Soal

## Soal 1

**Tingkat: Pemahaman**

Mengapa Wumpus World disebut *partially observable*? Sebutkan tiga aspek state
yang tidak diberikan secara langsung oleh sensor kepada agent.

<details>
<summary>Klik untuk melihat pembahasan</summary>

Wumpus World *partially observable* karena sensor hanya memberi petunjuk lokal,
bukan seluruh konfigurasi gua. Tiga contohnya adalah posisi agent, status hidup
Wumpus, dan apakah anak panah masih tersedia. Agent harus melacaknya menggunakan
percept serta riwayat aksi yang tersimpan di KB.

</details>

## Soal 2

**Tingkat: Penerapan**

Tuliskan PEAS untuk robot vacuum cleaner yang bergerak dalam ruangan. Setelah
itu, implementasikan fungsi `vacuum_percept(is_dirty, at_wall)` yang mengembalikan
`[Dirt, Bump]`; setiap elemen berisi nama sensornya saat aktif dan `None` saat
tidak aktif. Verifikasi fungsi dengan dua perintah berikut:

```python
assert vacuum_percept(True, False) == ["Dirt", None]
assert vacuum_percept(False, True) == [None, "Bump"]
```

<details>
<summary>Klik untuk melihat pembahasan</summary>

Salah satu definisi PEAS yang masuk akal adalah:

- **Performance measure:** banyaknya kotoran yang dibersihkan, waktu dan energi
  sesedikit mungkin, serta tanpa tabrakan.
- **Environment:** lantai, ruangan, dinding, furnitur, kotoran, dan tempat pengisian
  daya.
- **Actuators:** roda untuk maju/berbelok dan motor penyedot.
- **Sensors:** sensor kotoran dan benturan.

Implementasi yang memenuhi format percept:

```python
def vacuum_percept(is_dirty, at_wall):
    return ["Dirt" if is_dirty else None, "Bump" if at_wall else None]


assert vacuum_percept(True, False) == ["Dirt", None]
assert vacuum_percept(False, True) == [None, "Bump"]
```

PEAS lain dapat benar selama keempat komponennya konsisten dengan tujuan dan
cara kerja robot yang didefinisikan.

</details>

## Soal 3

**Tingkat: Analisis**

Agent berada di `[2,1]` dan merasakan Breeze. Mengapa agent tidak seharusnya
langsung melangkah ke `[2,2]`, walaupun `[2,2]` belum tentu berisi pit? Kaitkan
jawaban dengan informasi yang dimiliki agent dan performance measure.

<details>
<summary>Klik untuk melihat pembahasan</summary>

Breeze hanya memastikan bahwa sedikitnya satu kotak yang bersebelahan langsung
mengandung pit. Dari `[2,1]`, setelah `[1,1]` diketahui aman, kandidat yang tersisa
adalah `[2,2]` dan `[3,1]`. Informasi itu belum cukup untuk membuktikan `[2,2]`
aman. Memasukinya berarti menerima risiko kematian dengan penalti `-1000`, jauh
lebih buruk daripada biaya beberapa aksi tambahan sebesar `-1` per aksi untuk
kembali dan mencari informasi dari kotak aman `[1,2]`. Karena itu, agent rasional
lebih dahulu mengumpulkan percept tambahan.

</details>